# SURF Equipment Design and Cost Estimation


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch13_subsea_surf_systems\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch13_subsea_surf_systems\figures
NeqSim Java bridge available: True


## SURF Cost Breakdown


In [2]:
components = ["Trees", "Manifold", "Flowline", "Riser", "Umbilical", "Installation", "Commissioning"]
costs = np.array([80, 45, 310, 90, 210, 360, 120], dtype=float)
total = costs.sum()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(components, costs, color="#457b9d")
ax.set_ylabel("Cost (MNOK)")
ax.set_title(f"SURF Cost Breakdown, 4-Well 25 km Tieback (total {total:.0f} MNOK)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
fig.savefig(FIGURES_DIR / "ch13_surf_cost_breakdown.png", dpi=150, bbox_inches="tight")
plt.close(fig)
for name, cost in zip(components, costs):
    print(f"{name:<14} {cost:7.0f} MNOK  {100 * cost / total:5.1f}%")


Trees               80 MNOK    6.6%
Manifold            45 MNOK    3.7%
Flowline           310 MNOK   25.5%
Riser               90 MNOK    7.4%
Umbilical          210 MNOK   17.3%
Installation       360 MNOK   29.6%
Commissioning      120 MNOK    9.9%


**Discussion (Figure ch13_surf_cost_breakdown.png).** *Observation.* Flowline, installation and umbilical dominate the reference SURF cost. *Mechanism.* Long-distance tiebacks spend most capital on route length and vessel days rather than on the trees themselves. *Implication.* A small reduction in tieback length can outweigh detailed savings on individual tree hardware. *Recommendation.* Run route, diameter and installation-vessel sensitivities before optimizing smaller equipment packages.


## Manifold Break-Even


In [3]:
wells = np.arange(1, 9)
single_tree = wells * (85 + 18)
manifold = 65 + wells * 72 + 35
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(wells, single_tree, "o-", label="Independent tree tie-ins")
ax.plot(wells, manifold, "s-", label="Shared manifold cluster")
ax.set_xlabel("Number of wells")
ax.set_ylabel("Relative subsea connection cost (MNOK)")
ax.set_title("Single Tree vs Manifold Cluster Cost")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch13_surf_manifold_breakeven.png", dpi=150, bbox_inches="tight")
plt.close(fig)
best = np.where(manifold < single_tree)[0]
print(f"Manifold becomes cheaper at {wells[best[0]]} wells" if len(best) else "No break-even in range")


Manifold becomes cheaper at 4 wells


**Discussion (Figure ch13_surf_manifold_breakeven.png).** *Observation.* The shared manifold option becomes cheaper once enough wells share common infrastructure. *Mechanism.* The manifold has a fixed cost penalty but reduces per-well connection and routing cost. *Implication.* Well-count uncertainty matters directly for concept architecture. *Recommendation.* Evaluate manifold economics over the P10/P50/P90 well-count range, not only the base case.


## Water-Depth Cost Multiplier


In [4]:
depth = np.array([100, 300, 700, 1200, 1800])
multiplier = np.array([1.0, 1.35, 1.95, 2.8, 4.2])
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(depth, multiplier, marker="o", linewidth=2, color="#e76f51")
ax.set_xlabel("Water depth (m)")
ax.set_ylabel("SURF cost multiplier")
ax.set_title("Water Depth as a SURF Cost Driver")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURES_DIR / "ch13_surf_depth_multiplier.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch13_surf_depth_multiplier.png).** *Observation.* Cost rises nonlinearly as water depth enters deepwater and ultra-deepwater regimes. *Mechanism.* Installation complexity, riser loads, control systems and material requirements all scale with depth. *Implication.* Deepwater discoveries need larger recoverable volume or a nearby host to pass screening. *Recommendation.* Use water-depth multipliers early to avoid underestimating frontier concept CAPEX.
